In [ ]:
import numpy as np
import pandas as pd
import statistics
import seaborn as sns
import os
import subprocess
import shutil
import plotly.express as px
import sys
sys.path.append('/gpfs/commons/home/mgarbulowski/homic_package/src') 
from homic import file_readers, kraken2, process_data, make_plots
from collections import Counter
from scipy import stats
import matplotlib.pyplot as plt

In [ ]:
# define paths
path = "/gpfs/commons/home/mgarbulowski/016_proj_shm/metagenomes_lib/final_blastn/"
db_path = "/gpfs/commons/home/mgarbulowski/016_proj_shm/ref_dbs/kraken/human_hg38"

# samples id 

all_samps = ["KP003","KP004","KP005","KP008","KP010","KP011","KP012","KP013","KP016","KP021",
             "KP024","KP025","KP026","KP027","KP029","KP033","KP035","KP037","KP038","KP040",
             "KP041","KP046","KP047","KP048","KP049","KP052"] # all batches


## loading metadata
meta = pd.read_csv("/gpfs/commons/home/mgarbulowski/016_proj_shm/metagenomes_lib/metadata_kp.csv")
meta = meta.loc[meta['id'].isin(all_samps)] # keep only the sequenced ones

# intersecting with previous work
path1 = "/gpfs/commons/home/mgarbulowski/homic_package/files/65sp.txt"

with open(path1, "r") as f:
    bl_65sp = [line.strip() for line in f if line.strip()]
bl_65sp = [s.replace('[','').replace(']','') for s in bl_65sp]

# genera
gsg = bl_65sp
gsg = set([s.split()[0] for s in gsg])

In [ ]:

df_eed = pd.DataFrame({
    "no": list(range(1, 27)),
    "id": [
        "KP003","KP004","KP005","KP008","KP010","KP011","KP012","KP013",
        "KP016","KP021","KP024","KP025","KP026","KP027","KP029","KP033",
        "KP035","KP037","KP038","KP040","KP041","KP046","KP047","KP048",
        "KP049","KP052"
    ],
    "eed": [
        "yes","yes","yes","yes","yes","yes","yes","yes",
        "yes","yes","yes","no","no","no","no","no",
        "no","no","yes","no","no","no","no","no",
        "no","no"
    ]
})

df_eed = df_eed.loc[df_eed['id'].isin(all_samps)] # keep only the sequenced ones
print(df_eed)

In [ ]:
### EED ###
meta_eed = df_eed.loc[df_eed['eed'] == "yes"]
samps_ids = meta_eed["id"]
print(samps_ids)
path_blast = path
eed01, eed005, eed001, df_merged_eed = process_data.select_taxa(path_blast, samps_ids, taxa_level="species", db = "nt")

### NonEED ###
meta_need = df_eed.loc[df_eed['eed'] == "no"]
samps_ids = meta_need["id"]
path_blast = path
print(samps_ids)
need01, need005, need001, df_merged_need = process_data.select_taxa(path_blast, samps_ids, taxa_level="species", db = "nt")

In [ ]:
### female ###
meta_f = meta.loc[meta['female'] == 1]
samps_ids = meta_f["id"]
path_blast = path

fbc01, fbc005, fbc001, df_merged_f = process_data.select_taxa(path_blast, samps_ids, taxa_level="species", db = "nt")
print(df_merged_f)
### male ###
meta_m = meta.loc[meta['female'] == 0]
samps_ids = meta_m["id"]
path_blast = path
mbc01, mbc005, mbc001, df_merged_m = process_data.select_taxa(path_blast, samps_ids, taxa_level="species", db = "nt")

In [ ]:
### high_SES ###
meta_hs = meta.loc[meta['highSES'] == 1]
samps_ids = meta_hs["id"]
path_blast = path

hs01, hs005, hs001, df_merged_hs = process_data.select_taxa(path_blast, samps_ids, taxa_level="species", db = "nt")
### male ###
meta_ls = meta.loc[meta['highSES'] == 0]
samps_ids = meta_ls["id"]
path_blast = path
ls01, ls005, ls001, df_merged_ls = process_data.select_taxa(path_blast, samps_ids, taxa_level="species", db = "nt")

In [ ]:
# female
print("female")
print("0.1%")
print(len(fbc01))
print("")
print("0.05%")
print(len(fbc005))
print("")
print("0.01%")
print(len(fbc001))

# male
print("")
print("male")
print("0.1%")
print(len(mbc01))
print("")
print("0.05%")
print(len(mbc005))
print("")
print("0.01%")
print(len(mbc001))

# (>0.1%, >0.05% and >0.01% corresponding to 65, 121 and 419 species, respectively)

In [ ]:
ntop = 10
## species level female
vals_f = fbc01.values.tolist()
inds_f = fbc01.index.tolist()
inds_f2 = [name if val >= vals_f[ntop-1] else "Other" for name, val in zip(inds_f, vals_f)]

df_s_f = pd.DataFrame({'abundance': vals_f},
                  index=inds_f2)

## species level female
vals_m = mbc01.values.tolist()
inds_m = mbc01.index.tolist()
inds_m2 = [name if val >= vals_m[ntop-1] else "Other" for name, val in zip(inds_m, vals_m)]

df_s_m = pd.DataFrame({'abundance': vals_m},
                  index=inds_m2)

df_s_f = df_s_f.groupby(df_s_f.index).sum()
df_s_m = df_s_m.groupby(df_s_m.index).sum()

fig = make_plots.relative_abundance_double(df_s_f, df_s_m,"female","male")
fig.savefig("/gpfs/commons/home/mgarbulowski/016_proj_shm/metagenomes_lib/final_blastn/figs/relative_abundance_species_malefemale_0.1%_nt.png", dpi=300, bbox_inches='tight')


print("common species with previous work (female): ")
print(str(round(len(list(set(inds_f).intersection(bl_65sp)))/len(bl_65sp) * 100)) + "%" + " overlap")
print("")
print("common species with previous work (male): ")
print(str(round(len(list(set(inds_m).intersection(bl_65sp)))/len(bl_65sp) * 100)) + "%" + " overlap")


In [ ]:
ntop = 10
## species level EED
vals_f = eed01.values.tolist()
inds_f = eed01.index.tolist()
inds_f2 = [name if val >= vals_f[ntop-1] else "Other" for name, val in zip(inds_f, vals_f)]

df_s_f = pd.DataFrame({'abundance': vals_f},
                  index=inds_f2)

## species level NonEED
vals_m = need01.values.tolist()
inds_m = need01.index.tolist()
inds_m2 = [name if val >= vals_m[ntop-1] else "Other" for name, val in zip(inds_m, vals_m)]

df_s_m = pd.DataFrame({'abundance': vals_m},
                  index=inds_m2)

df_s_f = df_s_f.groupby(df_s_f.index).sum()
df_s_m = df_s_m.groupby(df_s_m.index).sum()

fig = make_plots.relative_abundance_double(df_s_f, df_s_m,"EED","non-EED")
fig.savefig("/gpfs/commons/home/mgarbulowski/016_proj_shm/metagenomes_lib/final_blastn/figs/relative_abundance_species_eed_need_0.1%_nt.png", dpi=300, bbox_inches='tight')

print("common species with previous work (female): ")
print(str(round(len(list(set(inds_f).intersection(bl_65sp)))/len(bl_65sp) * 100)) + "%" + " overlap")
print("")
print("common species with previous work (male): ")
print(str(round(len(list(set(inds_m).intersection(bl_65sp)))/len(bl_65sp) * 100)) + "%" + " overlap")


In [ ]:
ntop = 10
## species level EED
vals_f = hs01.values.tolist()
inds_f = hs01.index.tolist()
inds_f2 = [name if val >= vals_f[ntop-1] else "Other" for name, val in zip(inds_f, vals_f)]

df_s_f = pd.DataFrame({'abundance': vals_f},
                  index=inds_f2)

## species level NonEED
vals_m = ls01.values.tolist()
inds_m = ls01.index.tolist()
inds_m2 = [name if val >= vals_m[ntop-1] else "Other" for name, val in zip(inds_m, vals_m)]

df_s_m = pd.DataFrame({'abundance': vals_m},
                  index=inds_m2)

df_s_f = df_s_f.groupby(df_s_f.index).sum()
df_s_m = df_s_m.groupby(df_s_m.index).sum()

fig = make_plots.relative_abundance_double(df_s_f, df_s_m,"high SES","low SES")
fig.savefig("/gpfs/commons/home/mgarbulowski/016_proj_shm/metagenomes_lib/final_blastn/figs/relative_abundance_species_highlowSES_0.1%_nt.png", dpi=300, bbox_inches='tight')

print("common species with previous work (female): ")
print(str(round(len(list(set(inds_f).intersection(bl_65sp)))/len(bl_65sp) * 100)) + "%" + " overlap")
print("")
print("common species with previous work (male): ")
print(str(round(len(list(set(inds_m).intersection(bl_65sp)))/len(bl_65sp) * 100)) + "%" + " overlap")


In [ ]:
listed_data = [df_merged_m, df_merged_f]
counts = [Counter(df["genus"]) for df in listed_data]
data_names = ["male","female"]
# create DataFrame from all counts
df = pd.DataFrame(counts).fillna(0).astype(int)
df = df.T
df.columns = data_names

df['male']= (df['male']/df['male'].sum()) 
df['female']= (df['female']/df['female'].sum()) 

# 1
g = sns.regplot(x=data_names[0], y=data_names[1], data=df, scatter_kws={'s':20, 'color': 'black'}, line_kws={'color': 'black'})
res = stats.pearsonr(df[data_names[0]], df[data_names[1]])
    
g.text(
    0.05, 0.95,
    f"r = {res[0]:.2f}\n"
    f"p = {res[1]:.3e}",
    transform=g.transAxes,
    ha="left", va="top",
    fontsize=12, weight="bold"
    )

In [ ]:
listed_data = [df_merged_eed, df_merged_need]
counts = [Counter(df["genus"]) for df in listed_data]
data_names = ["EED","non-EED"]
# create DataFrame from all counts
df = pd.DataFrame(counts).fillna(0).astype(int)
df = df.T
df.columns = data_names

df['EED']= (df['EED']/df['EED'].sum()) 
df['non-EED']= (df['non-EED']/df['non-EED'].sum()) 

# 1
g = sns.regplot(x=data_names[0], y=data_names[1], data=df, scatter_kws={'s':20, 'color': 'black'}, line_kws={'color': 'black'})
res = stats.pearsonr(df[data_names[0]], df[data_names[1]])
    
g.text(
    0.05, 0.95,
    f"r = {res[0]:.2f}\n"
    f"p = {res[1]:.3e}",
    transform=g.transAxes,
    ha="left", va="top",
    fontsize=12, weight="bold"
    )

In [ ]:
listed_data = [df_merged_hs, df_merged_ls]
counts = [Counter(df["genus"]) for df in listed_data]
data_names = ["high SES","low SES"]
# create DataFrame from all counts
df = pd.DataFrame(counts).fillna(0).astype(int)
df = df.T
df.columns = data_names

df['high SES']= (df['high SES']/df['high SES'].sum()) 
df['low SES']= (df['low SES']/df['low SES'].sum()) 

# 1
g = sns.regplot(x=data_names[0], y=data_names[1], data=df, scatter_kws={'s':20, 'color': 'black'}, line_kws={'color': 'black'})
res = stats.pearsonr(df[data_names[0]], df[data_names[1]])
    
g.text(
    0.05, 0.95,
    f"r = {res[0]:.2f}\n"
    f"p = {res[1]:.3e}",
    transform=g.transAxes,
    ha="left", va="top",
    fontsize=12, weight="bold"
    )

In [ ]:
## genus level (based on species)
#female
s1 = fbc01.index.to_list()
s2 = [s.split()[0] for s in s1]

gfbc01 = fbc01.set_axis(s2, axis='index')
gfbc01 = gfbc01.groupby(level=0).sum()
gfbc01 = gfbc01.sort_values(ascending=False)
#male
s1 = mbc01.index.to_list()
s2 = [s.split()[0] for s in s1]

gmbc01 = mbc01.set_axis(s2, axis='index')
gmbc01 = gmbc01.groupby(level=0).sum()
gmbc01 = gmbc01.sort_values(ascending=False)

ntop = 10
## species level female
vals_f = gfbc01.values.tolist()
inds_f = gfbc01.index.tolist()
inds_f2 = [name if val >= vals_f[ntop-1] else "Other" for name, val in zip(inds_f, vals_f)]

df_s_f = pd.DataFrame({'abundance': vals_f},
                  index=inds_f2)

## species level female
vals_m = gmbc01.values.tolist()
inds_m = gmbc01.index.tolist()
inds_m2 = [name if val >= vals_m[ntop-1] else "Other" for name, val in zip(inds_m, vals_m)]

df_s_m = pd.DataFrame({'abundance': vals_m},
                  index=inds_m2)

df_s_f = df_s_f.groupby(df_s_f.index).sum()
df_s_m = df_s_m.groupby(df_s_m.index).sum()

fig = make_plots.relative_abundance_double(df_s_f, df_s_m,"female","male")
fig.savefig("/gpfs/commons/home/mgarbulowski/016_proj_shm/metagenomes_lib/final_blastn/figs/relative_abundance_genus_malefemale_0.1%_nt.png", dpi=300, bbox_inches='tight')

s1 = gmbc01.index.to_list()
s2 = [s.split()[0] for s in s1]
print("common genera with previous work (male): ")
print(str(round(len(list(set(s2).intersection(gsg)))/len(gsg) * 100)) + "%" + " overlap")

print("common genera with previous work (female): ")
s1 = gfbc01.index.to_list()
s2 = [s.split()[0] for s in s1]
print(str(round(len(list(set(s2).intersection(gsg)))/len(gsg) * 100)) + "%" + " overlap")



In [ ]:
## genus level (based on species)
#eed
s1 = eed01.index.to_list()
s2 = [s.split()[0] for s in s1]

gfbc01 = eed01.set_axis(s2, axis='index')
gfbc01 = gfbc01.groupby(level=0).sum()
gfbc01 = gfbc01.sort_values(ascending=False)
#non-eed
s1 = need01.index.to_list()
s2 = [s.split()[0] for s in s1]

gmbc01 = need01.set_axis(s2, axis='index')
gmbc01 = gmbc01.groupby(level=0).sum()
gmbc01 = gmbc01.sort_values(ascending=False)

ntop = 10
## species level female
vals_f = gfbc01.values.tolist()
inds_f = gfbc01.index.tolist()
inds_f2 = [name if val >= vals_f[ntop-1] else "Other" for name, val in zip(inds_f, vals_f)]

df_s_f = pd.DataFrame({'abundance': vals_f},
                  index=inds_f2)

## species level female
vals_m = gmbc01.values.tolist()
inds_m = gmbc01.index.tolist()
inds_m2 = [name if val >= vals_m[ntop-1] else "Other" for name, val in zip(inds_m, vals_m)]

df_s_m = pd.DataFrame({'abundance': vals_m},
                  index=inds_m2)

df_s_f = df_s_f.groupby(df_s_f.index).sum()
df_s_m = df_s_m.groupby(df_s_m.index).sum()

fig = make_plots.relative_abundance_double(df_s_f, df_s_m,"EED","non-EED")
fig.savefig("/gpfs/commons/home/mgarbulowski/016_proj_shm/metagenomes_lib/final_blastn/figs/relative_abundance_genus_eed_noneed_0.1%_nt.png", dpi=300, bbox_inches='tight')

s1 = gmbc01.index.to_list()
s2 = [s.split()[0] for s in s1]
print("common genera with previous work (eed): ")
print(str(round(len(list(set(s2).intersection(gsg)))/len(gsg) * 100)) + "%" + " overlap")

print("common genera with previous work (non-eed): ")
s1 = gfbc01.index.to_list()
s2 = [s.split()[0] for s in s1]
print(str(round(len(list(set(s2).intersection(gsg)))/len(gsg) * 100)) + "%" + " overlap")



In [ ]:
## genus level (based on species)
# high SES
s1 = hs01.index.to_list()
s2 = [s.split()[0] for s in s1]

gfbc01 = hs01.set_axis(s2, axis='index')
gfbc01 = gfbc01.groupby(level=0).sum()
gfbc01 = gfbc01.sort_values(ascending=False)
# low SES
s1 = ls01.index.to_list()
s2 = [s.split()[0] for s in s1]

gmbc01 = ls01.set_axis(s2, axis='index')
gmbc01 = gmbc01.groupby(level=0).sum()
gmbc01 = gmbc01.sort_values(ascending=False)

ntop = 10
## species level female
vals_f = gfbc01.values.tolist()
inds_f = gfbc01.index.tolist()
inds_f2 = [name if val >= vals_f[ntop-1] else "Other" for name, val in zip(inds_f, vals_f)]

df_s_f = pd.DataFrame({'abundance': vals_f},
                  index=inds_f2)

## species level female
vals_m = gmbc01.values.tolist()
inds_m = gmbc01.index.tolist()
inds_m2 = [name if val >= vals_m[ntop-1] else "Other" for name, val in zip(inds_m, vals_m)]

df_s_m = pd.DataFrame({'abundance': vals_m},
                  index=inds_m2)

df_s_f = df_s_f.groupby(df_s_f.index).sum()
df_s_m = df_s_m.groupby(df_s_m.index).sum()

fig = make_plots.relative_abundance_double(df_s_f, df_s_m,"high SES","low SES")
fig.savefig("/gpfs/commons/home/mgarbulowski/016_proj_shm/metagenomes_lib/final_blastn/figs/relative_abundance_genus_highlowSES_0.1%_nt.png", dpi=300, bbox_inches='tight')

s1 = gmbc01.index.to_list()
s2 = [s.split()[0] for s in s1]
print("common genera with previous work (eed): ")
print(str(round(len(list(set(s2).intersection(gsg)))/len(gsg) * 100)) + "%" + " overlap")

print("common genera with previous work (non-eed): ")
s1 = gfbc01.index.to_list()
s2 = [s.split()[0] for s in s1]
print(str(round(len(list(set(s2).intersection(gsg)))/len(gsg) * 100)) + "%" + " overlap")



In [ ]:
listed_data = [df_merged_m, df_merged_f]
counts = [Counter(df["genus"]) for df in listed_data]
data_names = ["male","female"]
# create DataFrame from all counts
df = pd.DataFrame(counts).fillna(0).astype(int)
df = df.T
df.columns = data_names

df['male']= (df['male']/df['male'].sum()) 
df['female']= (df['female']/df['female'].sum()) 

# 1
g = sns.regplot(x=data_names[0], y=data_names[1], data=df, scatter_kws={'s':20, 'color': 'black'}, line_kws={'color': 'black'})
res = stats.pearsonr(df[data_names[0]], df[data_names[1]])
    
g.text(
    0.05, 0.95,
    f"r = {res[0]:.2f}\n"
    f"p = {res[1]:.3e}",
    transform=g.transAxes,
    ha="left", va="top",
    fontsize=12, weight="bold"
    )

In [ ]:
## EED vs non-EED
listed_data = [df_merged_eed, df_merged_need]
counts = [Counter(df["genus"]) for df in listed_data]
data_names = ["EED","non-EED"]
# create DataFrame from all counts
df = pd.DataFrame(counts).fillna(0).astype(int)
df = df.T
df.columns = data_names

df['EED']= (df['EED']/df['EED'].sum()) 
df['non-EED']= (df['non-EED']/df['non-EED'].sum()) 

# 1
g = sns.regplot(x=data_names[0], y=data_names[1], data=df, scatter_kws={'s':20, 'color': 'black'}, line_kws={'color': 'black'})
res = stats.pearsonr(df[data_names[0]], df[data_names[1]])
    
g.text(
    0.05, 0.95,
    f"r = {res[0]:.2f}\n"
    f"p = {res[1]:.3e}",
    transform=g.transAxes,
    ha="left", va="top",
    fontsize=12, weight="bold"
    )

In [ ]:
## low vs high SES
listed_data = [df_merged_ls, df_merged_hs]
counts = [Counter(df["genus"]) for df in listed_data]
data_names = ["low SES","high SES"]
# create DataFrame from all counts
df = pd.DataFrame(counts).fillna(0).astype(int)
df = df.T
df.columns = data_names

df['low SES']= (df['low SES']/df['low SES'].sum()) 
df['high SES']= (df['high SES']/df['high SES'].sum()) 

# 1
g = sns.regplot(x=data_names[0], y=data_names[1], data=df, scatter_kws={'s':20, 'color': 'black'}, line_kws={'color': 'black'})
res = stats.pearsonr(df[data_names[0]], df[data_names[1]])
    
g.text(
    0.05, 0.95,
    f"r = {res[0]:.2f}\n"
    f"p = {res[1]:.3e}",
    transform=g.transAxes,
    ha="left", va="top",
    fontsize=12, weight="bold"
    )

In [ ]:
##### sample analysis ##########
import glob

samps_ids = all_samps

bac_no = []
bac_no_un = []
for idx, samp in enumerate(samps_ids):

    pattern = os.path.join(path_blast, samp + "*_blastn_report.txt")
    matches = glob.glob(pattern)
    if matches:
        path_blast_full = matches[0]  # or handle multiple matches
    else:
        path_blast_full = None

    df = process_data.read_n_clean_blastn(path_blast_full, db = "nt") # use defaults
    bac_no.append(len(df["subject_id"]))
    bac_no_un.append(df["subject_id"].nunique())


In [ ]:
import pandas as pd

def count_phages_and_viruses(file_path):
    # Load BLAST results (tab-separated file)
    df = pd.read_csv(file_path, sep="\t", header=None)

    # Assign column names
    df.columns = [
        "query", "subject", "identity", "length",
        "evalue", "bitscore", "qlen", "description"
    ]

    # Identify phages and viruses
    phage_mask = df["description"].str.contains("phage", case=False, na=False)
    virus_mask = df["description"].str.contains("virus", case=False, na=False)

    # Counts (avoid double counting phages as viruses)
    num_phages = phage_mask.sum()
    num_viruses_only = (virus_mask & ~phage_mask).sum()

    # Unique contigs
    phage_contigs = df[phage_mask]["query"].nunique()
    virus_contigs = df[virus_mask]["query"].nunique()

    return num_phages, num_viruses_only, phage_contigs, virus_contigs

In [ ]:
vir_no = []
pha_no = []
vir_no_un = []
pha_no_un = []
s_id = []
for idx, samp in enumerate(samps_ids):

    pattern = os.path.join(path_blast, samp + "*_blastn_report.txt")
    matches = glob.glob(pattern)
    if matches:
        path_blast_full = matches[0]  # or handle multiple matches
    else:
        path_blast_full = None

    phages, viruses, phage_un, virus_un = count_phages_and_viruses(path_blast_full)
    pha_no.append(phages)
    vir_no.append(viruses)
    pha_no_un.append(phage_un)
    vir_no_un.append(virus_un)
    s_id.append(samp)

pha_no = [int(x) for x in pha_no]
vir_no = [int(x) for x in vir_no]
pha_no_un = [int(x) for x in pha_no_un]
vir_no_un = [int(x) for x in vir_no_un]

In [ ]:
kin_stats = pd.DataFrame({
    "sample": s_id,
    "bacteria": bac_no_un,
    "virus": vir_no_un,
    "phage": pha_no_un,
    "eed": df_eed["eed"]
})


kin_stats["eed"] = kin_stats["eed"].replace({
    "yes": "eed",
    "no": "non-eed"
})

df = kin_stats

numeric_cols = ["bacteria", "virus", "phage"]

# Normalize by column sum
df_norm = df.copy()
df_norm[numeric_cols] = df[numeric_cols] / df[numeric_cols].sum()

# Convert to percentage (optional)
df_percent = df_norm.copy()
df_percent[numeric_cols] = df_percent[numeric_cols]*10

kin_stats = df_percent

In [ ]:
kin_stats = pd.DataFrame({
    "sample": s_id,
    "bacteria": bac_no_un,
    "virus": vir_no_un,
    "phage": pha_no_un,
    "eed": meta["highSES"]
})


kin_stats["eed"] = kin_stats["eed"].replace({
    "1": "high",
    "0": "low"
})

df = kin_stats

numeric_cols = ["bacteria", "virus", "phage"]

# Normalize by column sum
df_norm = df.copy()
df_norm[numeric_cols] = df[numeric_cols] / df[numeric_cols].sum()

# Convert to percentage (optional)
df_percent = df_norm.copy()
df_percent[numeric_cols] = df_percent[numeric_cols]*10

kin_stats = df_percent

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from statannotations.Annotator import Annotator

# reshape to long format
df_long = kin_stats.melt(
    id_vars="eed",
    value_vars=["bacteria", "virus", "phage"],
    var_name="kingdom",
    value_name="value"
)

# style
sns.set(style="whitegrid", context="talk")

plt.figure(figsize=(10, 6))

ax = sns.boxplot(
    data=df_long,
    x="kingdom",
    y="value",
    hue="eed",
    palette="Set2"
)

# legend
plt.legend(title="", bbox_to_anchor=(1.02, 1), loc="upper left")

# labels
plt.ylabel("Number of unique species")
plt.xlabel("")

# ---------------------------
# ⭐ ADD STATISTICAL TESTS
# ---------------------------

# get your group names automatically
groups = df_long["eed"].unique()

# define comparisons (assumes 2 groups; extend if needed)
pairs = [
    (("bacteria", groups[0]), ("bacteria", groups[1])),
    (("virus", groups[0]), ("virus", groups[1])),
    (("phage", groups[0]), ("phage", groups[1]))
]

annotator = Annotator(
    ax,
    pairs,
    data=df_long,
    x="kingdom",
    y="value",
    hue="eed"
)

annotator.configure(
    test="Mann-Whitney",   # better for microbiome data
    text_format="simple",    # *, **, ***
    loc="outside",
    verbose=0
)

annotator.apply_and_annotate()

# clean up
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
## abundance analysis
import os
import glob
import pandas as pd

all_dfs = []
samps_ids = all_samps

for idx, samp in enumerate(samps_ids):

    pattern = os.path.join(path_blast, samp + "*_blastn_report.txt")
    matches = glob.glob(pattern)

    if matches:
        path_blast_full = matches[0]
    else:
        path_blast_full = None

    if path_blast_full is None:
        continue

    df = process_data.read_n_clean_blastn(path_blast_full, db="nt")

    # IMPORTANT: add sample ID column
    df["sample"] = samp

    all_dfs.append(df)

# ---- combine all samples ----
combined_df = pd.concat(all_dfs, ignore_index=True)

# Assume species column is called "species"
# (rename here if needed)
# combined_df = combined_df.rename(columns={"your_species_col": "species"})

# ---- count occurrences per sample/species ----
counts = (
    combined_df
    .groupby(["sample", "genus"])
    .size()
    .reset_index(name="count")
)

# ---- convert to abundance (% per sample) ----
counts["abundance_pct"] = counts.groupby("sample")["count"].transform(
    lambda x: 100 * x / x.sum()
)

# ---- pivot into table (samples x species) ----
abundance_table = counts.pivot_table(
    index="sample",
    columns="genus",
    values="abundance_pct",
    fill_value=0
)

# optional: sort samples
abundance_table = abundance_table.sort_index()

print(abundance_table)

In [ ]:
from scipy.stats import spearmanr

spearman_results = []
case_control_vector = df_eed["eed"]
print(case_control_vector)
mapping = {"no": 0, "yes": 1}
y = np.array([mapping[v] for v in case_control_vector])


abundance_table2 = abundance_table.loc[:, (abundance_table == 0).mean() < 0.50]
for species in abundance_table2.columns:
    r, p = spearmanr(abundance_table2[species], y)
    spearman_results.append((species, r, p))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# your input list
results = spearman_results

df = pd.DataFrame(results, columns=["species", "corr", "pval"])

# top 10 positive + negative
df_sorted = df.sort_values("corr")
plot_df = pd.concat([df_sorted.head(10), df_sorted.tail(10)])
plot_df = plot_df.sort_values("corr")

# significance transform
plot_df["logp"] = -np.log10(plot_df["pval"])

# normalize for color intensity
norm = (plot_df["logp"] - plot_df["logp"].min()) / (plot_df["logp"].max() - plot_df["logp"].min() + 1e-9)

# color map: red for positive, blue for negative
colors = []
for corr, n in zip(plot_df["corr"], norm):
    if corr >= 0:
        colors.append((1, 0.3, 0.3, 0.3 + 0.7*n))  # red intensity
    else:
        colors.append((0.3, 0.3, 1, 0.3 + 0.7*n))  # blue intensity

plt.figure(figsize=(10, 6))
plt.barh(plot_df["species"], plot_df["corr"], color=colors)

plt.axvline(0, color="black", linewidth=1)
plt.xlabel("Spearman correlation")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
from bacdive import BacdiveClient

# Species list
species_list = list(plot_df["species"])
client = BacdiveClient()
client.setSearchType("exact")

def extract_field_safe(obj, key_path):
    """
    Recursively extract a field from nested dicts/lists.
    key_path: list of keys to traverse.
    Returns the first found value or None.
    """
    if obj is None:
        return None
    if not key_path:
        return None
    key = key_path[0]

    if isinstance(obj, dict):
        val = obj.get(key)
        if len(key_path) == 1:
            return val
        else:
            return extract_field_safe(val, key_path[1:])
    elif isinstance(obj, list):
        # iterate over list elements, return first non-None
        for item in obj:
            val = extract_field_safe(item, key_path)
            if val is not None:
                return val
        return None
    return None

# Collect all raw strain info
all_records = []

for name in species_list:
    print(f"Searching BacDive for: {name}")
    client.search(taxonomy=name)
    raw_results = list(client.retrieve())
    
    if not raw_results:
        all_records.append({
            "species": name,
            "BacDive_ID": None,
            "keywords": None,
            "description": None,
            "culture_collection": None,
            "sample_type": None,
            "geographic_location": None,
            "country": None,
            "biosafety_level": None,
            "human_pathogen": None,
            "pathogen_flag": "No data"
        })
        continue

    for strain in raw_results:
        record = {"species": name}

        # Extract fields safely
        record["BacDive_ID"] = extract_field_safe(strain, ["General", "BacDive-ID"])
        record["keywords"] = extract_field_safe(strain, ["General", "keywords"])
        record["description"] = extract_field_safe(strain, ["General", "description"])
        record["culture_collection"] = extract_field_safe(strain, ["Literature", "culture collection no."])
        record["sample_type"] = extract_field_safe(strain, ["Isolation, sampling and environmental information", "isolation", "sample type"])
        record["geographic_location"] = extract_field_safe(strain, ["Isolation, sampling and environmental information", "isolation", "geographic location"])
        record["country"] = extract_field_safe(strain, ["Isolation, sampling and environmental information", "isolation", "country"])
        record["biosafety_level"] = extract_field_safe(strain, ["Interaction and safety", "risk assessment", "biosafety level"])
        record["human_pathogen"] = extract_field_safe(strain, ["Interaction and safety", "risk assessment", "pathogenicity human"])

        all_records.append(record)

# Convert to DataFrame
df = pd.DataFrame(all_records)

# Flatten by species
def merge_series(series):
    """Merge lists or unique strings into a single string, ignore NaN"""
    series = series.dropna()
    values = set()
    for val in series:
        if isinstance(val, list):
            values.update(val)
        else:
            values.add(str(val))
    return "; ".join(values) if values else None

flat_df = df.groupby("species").agg({
    "BacDive_ID": merge_series,
    "keywords": merge_series,
    "description": merge_series,
    "culture_collection": merge_series,
    "sample_type": merge_series,
    "geographic_location": merge_series,
    "country": merge_series,
    "biosafety_level": lambda x: max([int(b) for b in x.dropna() if str(b).isdigit()]) if any(x.dropna()) else None,
    "human_pathogen": merge_series
}).reset_index()

# Recompute pathogen_flag for flat species overview
def compute_pathogen_flag(row):
    keywords = row["keywords"] or ""
    human_pathogen = row["human_pathogen"] or ""
    keywords_str = keywords.lower() if isinstance(keywords, str) else ""
    if "pathogen" in keywords_str or "yes" in str(human_pathogen).lower():
        return "Potential pathogen"
    else:
        return "Probably non-pathogenic"

flat_df["pathogen_flag"] = flat_df.apply(compute_pathogen_flag, axis=1)


# Show result
print(flat_df)


# Save to CSV if needed
#flat_df.to_csv(main_path + "/bacdive_mod+sev_EED_unique.csv", index=False)


In [ ]:
pathogens_df = flat_df[flat_df["pathogen_flag"] == "Potential pathogen"]
pathogen_species = pathogens_df["species"].tolist()
print(pathogen_species)

pathogens = pathogen_species

plt.figure(figsize=(10, 6))
plt.barh(plot_df["species"], plot_df["corr"], color=colors)

# y positions
y_pos = range(len(plot_df))

for i, (species, corr) in enumerate(zip(plot_df["species"], plot_df["corr"])):
    if species in pathogens:
        plt.scatter(
            corr,
            i,
            marker="D",   # diamond marker
            s=80,
            color = "orange",
            edgecolor="orange"
        )

plt.axvline(0, color="black", linewidth=1)
plt.xlabel("Spearman correlation")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# =========================
# INPUTS (you already have these)
# =========================

# 1. correlation results: list of tuples
# (species, correlation, p-value)
results = spearman_results


# =========================
# STEP 1: Build correlation dataframe
# =========================

df_corr = pd.DataFrame(results, columns=["species", "corr", "pval"])

# =========================
# STEP 2: Select top species
# =========================

df_sorted = df_corr.sort_values("corr")

top_n = 10
top_neg = df_sorted.head(top_n)
top_pos = df_sorted.tail(top_n)

plot_df = pd.concat([top_neg, top_pos]).sort_values("corr")

selected_species = plot_df["species"].tolist()

# =========================
# STEP 3 (OPTIONAL): Filter only pathogens
# =========================

# Uncomment if you have pathogen_flag
# plot_df = plot_df.merge(df_full[["species", "pathogen_flag"]], on="species", how="left")
# plot_df = plot_df[plot_df["pathogen_flag"] == "Potential pathogen"]
# selected_species = plot_df["species"].tolist()
print(case_control_vector)
metadata = pd.DataFrame({
    "case_control": list(case_control_vector)
}, index=abundance_table.index)

row_colors = metadata["case_control"].map({
    "yes": "darkviolet",
    "no": "limegreen"
})
print(row_colors)
# =========================
# STEP 4: Subset abundance table
# =========================

heatmap_df = abundance_table[selected_species]

# --- ensure numeric ---
heatmap_df = heatmap_df.astype(float)

# --- mask zeros ---
zero_mask = heatmap_df == 0

# --- log transform ---
heatmap_log = np.log10(heatmap_df + 1e-6)

# --- scale ---
heatmap_scaled = heatmap_log.apply(
    lambda x: (x - x.mean()) / (x.std() + 1e-9),
    axis=0
)

# --- clustering data (no NaN) ---
heatmap_for_clustering = heatmap_scaled.fillna(heatmap_scaled.min().min())

# --- display data (with NaN) ---
heatmap_display = heatmap_scaled.copy()
heatmap_display[zero_mask] = np.nan

# --- colormap ---
cmap = plt.cm.coolwarm.copy()
cmap.set_bad(color="slategrey")

# --- plot FIRST (with clustering data) ---
g = sns.clustermap(
    heatmap_for_clustering,
    cmap=cmap,
    row_colors=row_colors,
    figsize=(14, 12),
    cbar_pos=(0.02, 0.02, 0.03, 0.2)
)

# --- get clustered order ---
row_order = g.dendrogram_row.reordered_ind
col_order = g.dendrogram_col.reordered_ind

# --- reorder display data to match clustering ---
heatmap_display_reordered = heatmap_display.iloc[row_order, col_order]

# --- redraw heatmap with correct data ---
g.ax_heatmap.clear()
sns.heatmap(
    heatmap_display_reordered,
    cmap=cmap,
    ax=g.ax_heatmap,
    cbar=True,
    cbar_ax=g.cax,
    linewidths=0.5,          # 👈 thickness of lines
    linecolor="white"        # 👈 color of lines
)
g.ax_heatmap.yaxis.tick_right()
g.ax_heatmap.yaxis.set_label_position("right")
plt.savefig("/gpfs/commons/home/mgarbulowski/016_proj_shm/metagenomes_lib/final_blastn/figs/spearman_correlation_abundance_heatmap.png", dpi=300, bbox_inches='tight')
plt.show()



In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# =========================
# INPUTS (YOU ALREADY HAVE THESE)
# =========================

# abundance_table: rows = samples, columns = species
# case_control_vector: ["yes", "no", ...]
# selected_species: list of species (e.g. correlated ones)

# =========================
# STEP 1: ALIGN + CLEAN DATA
# =========================

# ensure alignment safety
abundance_table = abundance_table.copy()
abundance_table = abundance_table.astype(float)

df = abundance_table.copy()
df["group"] = list(case_control_vector)

#selected_species = list(abundance_table.columns)
#print(selected_species)
selected_species = ["Turicibacter"]

df_long = df.melt(
    id_vars="group",
    var_name="species",
    value_name="abundance"
)

# normalize group labels (safety)
df_long["group"] = df_long["group"].str.lower()

# keep only valid groups
df_long = df_long[df_long["group"].isin(["yes", "no"])]

# =========================
# STEP 2: FILTER VALID SPECIES
# =========================

selected_species = [s for s in selected_species if s in abundance_table.columns]

if len(selected_species) == 0:
    raise ValueError("No valid species found in abundance_table.")

# =========================
# STEP 3: STATISTICS (SAFE LOOP)
# =========================

results = []

for sp in selected_species:

    case_vals = df_long[(df_long["species"] == sp) & (df_long["group"] == "yes")]["abundance"]
    ctrl_vals = df_long[(df_long["species"] == sp) & (df_long["group"] == "no")]["abundance"]

    # skip invalid cases
    if len(case_vals) < 2 or len(ctrl_vals) < 2:
        continue

    # skip all-zero species
    if case_vals.sum() == 0 and ctrl_vals.sum() == 0:
        continue

    stat, p = mannwhitneyu(case_vals, ctrl_vals, alternative="two-sided")

    results.append((sp, p))

# =========================
# STEP 4: BUILD STATS TABLE
# =========================

stats_df = pd.DataFrame(results, columns=["species", "pval"])
print(stats_df)

if len(stats_df) == 0:
    raise ValueError("No valid statistical comparisons could be computed.")

# FDR correction
print(stats_df["pval"])
stats_df["p_adj"] = multipletests(stats_df["pval"], method="fdr_bh")[1]

# =========================
# STEP 5: ORDER SPECIES
# =========================

species_order = stats_df.sort_values("p_adj")["species"].tolist()

# =========================
# STEP 6: PREPARE PLOTTING DATA
# =========================

plot_df = df_long[df_long["species"].isin(species_order)].copy()
plot_df = df_long[df_long["species"].isin(selected_species)]
# =========================
# STEP 7: PLOT BOX + POINTS
# =========================

plt.figure(figsize=(5, 8))

sns.boxplot(
    data=plot_df,
    x="species",
    y="abundance",
    hue="group",
    order=species_order
)

sns.stripplot(
    data=plot_df,
    x="species",
    y="abundance",
    hue="group",
    order=species_order,
    dodge=True,
    alpha=0.5,
    color="black"
)

# =========================
# STEP 8: SIGNIFICANCE LABELS
# =========================

y_max = plot_df["abundance"].max()

for i, sp in enumerate(species_order):

    pvals = stats_df.loc[stats_df["species"] == sp, "p_adj"].values
    print(pvals)
    if len(pvals) == 0:
        continue

    p = pvals[0]

    if p < 0.01:
        label = "***"
    elif p < 0.05:
        label = "**"
    elif p < 0.1:
        label = "*"
    else:
        label = "ns"

    plt.text(i, y_max * 1.05, label, ha="center", fontsize=10)

# =========================
# FINAL CLEANUP
# =========================

plt.xticks(rotation=90)
plt.ylabel("Relative abundance")
plt.xlabel("Species")
#plt.title("Case vs Control Differential Abundance (Wilcoxon + FDR)")

# fix duplicate legends
handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(handles[:2], labels[:2], title="EED", bbox_to_anchor=(1.05, 1), loc=2)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# =========================
# USER PARAMETER
# =========================
p_threshold = 0.05  # <-- change this as you like

# =========================
# STEP 1: ALIGN + CLEAN DATA
# =========================

abundance_table = abundance_table.copy().astype(float)

df = abundance_table.copy()

#df["group"] = list(case_control_vector)

df["group"] = list(meta['highSES'])
df["group"] = list(df["group"].map({0: 'lowSES', 1: 'highSES'}))


selected_species = list(abundance_table.columns)

df_long = df.melt(
    id_vars="group",
    var_name="species",
    value_name="abundance"
)

df_long["group"] = df_long["group"].str.lower()
df_long = df_long[df_long["group"].isin(["yes", "no"])]

# =========================
# STEP 2: FILTER VALID SPECIES
# =========================

selected_species = [s for s in selected_species if s in abundance_table.columns]

if len(selected_species) == 0:
    raise ValueError("No valid species found in abundance_table.")

# =========================
# STEP 3: STATISTICS
# =========================

results = []

for sp in selected_species:

    case_vals = df_long[(df_long["species"] == sp) & (df_long["group"] == "yes")]["abundance"]
    ctrl_vals = df_long[(df_long["species"] == sp) & (df_long["group"] == "no")]["abundance"]

    if len(case_vals) < 2 or len(ctrl_vals) < 2:
        continue

    if case_vals.sum() == 0 and ctrl_vals.sum() == 0:
        continue

    stat, p = mannwhitneyu(case_vals, ctrl_vals, alternative="two-sided")
    results.append((sp, p))

# =========================
# STEP 4: BUILD STATS TABLE
# =========================

stats_df = pd.DataFrame(results, columns=["species", "pval"])

if len(stats_df) == 0:
    raise ValueError("No valid statistical comparisons could be computed.")

# FDR correction
#stats_df["p_adj"] = multipletests(stats_df["pval"], method="fdr_bh")[1]
stats_df["p_adj"] = stats_df["pval"]
# =========================
# STEP 5: FILTER BY P-VALUE
# =========================

stats_df_sig = stats_df[stats_df["p_adj"] < p_threshold].copy()

if len(stats_df_sig) == 0:
    raise ValueError("No species pass the chosen p-value threshold.")

# Order species by significance
species_order = stats_df_sig.sort_values("p_adj")["species"].tolist()

# =========================
# STEP 6: PREPARE PLOTTING DATA
# =========================

plot_df = df_long[df_long["species"].isin(species_order)].copy()

# =========================
# STEP 7: PLOT
# =========================

plt.figure(figsize=(max(6, len(species_order) * 0.6), 6))

sns.boxplot(
    data=plot_df,
    x="species",
    y="abundance",
    hue="group",
    order=species_order
)

sns.stripplot(
    data=plot_df,
    x="species",
    y="abundance",
    hue="group",
    order=species_order,
    dodge=True,
    alpha=0.5,
    color="black"
)

# =========================
# STEP 8: SIGNIFICANCE LABELS
# =========================

y_max = plot_df["abundance"].max()

for i, sp in enumerate(species_order):

    p = stats_df_sig.loc[stats_df_sig["species"] == sp, "p_adj"].values[0]

    if p < 0.01:
        label = "***"
    elif p < 0.05:
        label = "**"
    elif p < 0.1:
        label = "*"
    else:
        continue  # skip non-significant (shouldn't happen due to filtering)

    plt.text(i, y_max * 1.05, label, ha="center", fontsize=10)

# =========================
# FINAL CLEANUP
# =========================

plt.xticks(rotation=90)
plt.ylabel("Relative abundance")
plt.xlabel("Species")

# Fix duplicate legends
handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(handles[:2], labels[:2], title="Group", bbox_to_anchor=(1.05, 1), loc=2)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr, mannwhitneyu, kruskal
from statsmodels.stats.multitest import multipletests

# =========================
# INPUTS
# =========================
# abundance_table: rows = samples, columns = species
# metadata: rows = samples, columns = variables

abundance_table = abundance_table.copy().astype(float)
metadata = meta.copy()

# align samples

metadata.index = metadata["id"]
metadata = metadata.loc[abundance_table.index]
#metadata["eed"] = list(df_eed["eed"])
#print(metadata)
# =========================
# SIGNIFICANCE FUNCTION
# =========================

def p_to_stars(p):
    if p < 0.01:
        return "***"
    elif p < 0.05:
        return "**"
    elif p < 0.1:
        return "*"
    else:
        return "ns"

# =========================
# TEST FUNCTION
# =========================

def test_variable(var_name, meta_col, abundance_table):

    results = []

    # detect variable type
    if pd.api.types.is_numeric_dtype(meta_col):
        var_type = "numeric"
    else:
        var_type = "categorical"

    for sp in abundance_table.columns:

        abund = abundance_table[sp]

        # align + drop missing
        valid_idx = meta_col.notna()
        abund = abund[valid_idx]
        meta = meta_col[valid_idx]

        if len(abund) < 3:
            continue

        try:
            # =====================
            # NUMERIC → Spearman
            # =====================
            if var_type == "numeric":

                corr, p = spearmanr(abund, meta)
                stat = corr

            # =====================
            # CATEGORICAL
            # =====================
            else:
                groups = meta.unique()

                if len(groups) == 2:

                    g1 = abund[meta == groups[0]]
                    g2 = abund[meta == groups[1]]

                    if len(g1) < 2 or len(g2) < 2:
                        continue

                    stat, p = mannwhitneyu(g1, g2)

                elif len(groups) > 2:

                    group_vals = [abund[meta == g] for g in groups]

                    if any(len(g) < 2 for g in group_vals):
                        continue

                    stat, p = kruskal(*group_vals)

                else:
                    continue

            results.append((var_name, sp, stat, p, var_type))

        except Exception:
            continue

    return results

# =========================
# RUN ALL VARIABLES
# =========================

all_results = []

for col in metadata.columns:
    all_results.extend(
        test_variable(col, metadata[col], abundance_table)
    )

results_df = pd.DataFrame(
    all_results,
    columns=["variable", "species", "stat", "pval", "type"]
)

if results_df.empty:
    raise ValueError("No valid statistical comparisons were computed.")

# =========================
# FDR CORRECTION (FIXED)
# =========================

# force numeric + remove invalid p-values
results_df["pval"] = pd.to_numeric(results_df["pval"], errors="coerce")

mask = results_df["pval"].notna() & np.isfinite(results_df["pval"])

results_df.loc[mask, "p_adj"] = multipletests(
    results_df.loc[mask, "pval"],
    method="fdr_bh"
)[1]

# optional: keep structure clean
results_df.loc[~mask, "p_adj"] = np.nan
# =========================
# SIGNIFICANCE LABELS
# =========================

results_df["significance"] = results_df["p_adj"].apply(p_to_stars)

# =========================
# EFFECT SIZE (for ranking)
# =========================

results_df["effect_size"] = results_df.apply(
    lambda x: abs(x["stat"]) if x["type"] == "numeric" else x["stat"],
    axis=1
)

# =========================
# SORT RESULTS
# =========================

results_df = results_df.sort_values(
    ["p_adj", "effect_size"],
    ascending=[True, False]
)

# =========================
# CLEAN OUTPUT ORDER
# =========================

results_df = results_df[[
    "variable",
    "species",
    "stat",
    "pval",
    "p_adj",
    "significance",
    "type"
]]
sig_df = results_df[results_df["pval"] < 0.05]
print(sig_df)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import numpy as np

colors = plt.cm.tab20(np.linspace(0, 1, 15))
# =========================
# COPY + CLEAN DATA
# =========================
df = sig_df.copy()

# rename variables
df["variable"] = df["variable"].replace({
    "female": "sex",
    "highSES": "SES"
})
print(df["variable"])
# =========================
# 1. PIE CHART: VARIABLES
# =========================
var_counts = df["variable"].value_counts()

plt.figure()
plt.pie(
    var_counts.values,
    labels=var_counts.index,
    autopct="%1.1f%%",
    colors=colors,
    textprops={"fontsize": 10}
)
plt.title("Most Frequently Affected Variables")
plt.show()


# =========================
# 2. PIE CHART: SPECIES
# =========================
sp_counts = df["species"].value_counts().head(10)

plt.figure()
plt.pie(
    sp_counts.values,
    labels=sp_counts.index,
    autopct="%1.1f%%",
    colors=colors,
    textprops={"fontsize": 10}
)
plt.title("Most Frequently Associated Species (Top 10)")
plt.show()


# =========================
# 3. VARIABLE SUMMARY
# =========================
df_var = df.groupby("variable").agg(
    freq=("species", "count"),
    mean_effect=("stat", "mean"),
    abs_effect=("stat", lambda x: np.mean(np.abs(x)))
).reset_index()

plt.figure()
plt.scatter(df_var["freq"], df_var["abs_effect"])

for _, row in df_var.iterrows():
    plt.text(row["freq"], row["abs_effect"], row["variable"])

plt.xlabel("Number of Associations")
plt.ylabel("Mean Absolute Effect Size")
plt.title("Variable Frequency vs Effect Size")
plt.show()


# =========================
# TOP ASSOCIATIONS (clean view)
# =========================
import numpy as np
import matplotlib.pyplot as plt

top_n = 20
df_top = df.reindex(df["stat"].abs().sort_values(ascending=False).index).head(top_n)

# create labels like "variable → species"
labels = df_top["variable"] + " → " + df_top["species"]

colors = np.where(df_top["stat"] > 0, "steelblue", "tomato")

plt.figure(figsize=(10, 6))
plt.barh(labels[::-1], df_top["stat"][::-1], color=colors[::-1])

plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Effect Size (stat)")
plt.title("Top Microbe–Variable Associations (Effect Direction Included)")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================
# TOP N STRONGEST HITS
# =========================
top_n = 20

df_top = df.copy()

# label
df_top["label"] = df_top["variable"] + " → " + df_top["species"]

# STEP 1: take top N by absolute effect size
df_top = df_top.reindex(df_top["stat"].abs().sort_values(ascending=False).index).head(top_n)

# STEP 2: now group by variable (within top N only)
df_top = df_top.sort_values(["variable", "stat"], ascending=[True, False])

labels = df_top["label"]
values = df_top["stat"]

colors = np.where(values > 0, "tomato", "steelblue")

plt.figure(figsize=(12, 8))
plt.barh(labels, values, color=colors)

plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Effect Size (correlation)")
#plt.title("Top Microbe–Variable Associations (Grouped within Top N)")
plt.tight_layout()
plt.show()